# 9. A cell line the model never saw: HEK293 (Fig. 4e)

HEK293 is not among the 116 EpiMap biosamples. Its promoter windows were built with the same rules as the EpiMap windows (the TSS window plus up to nine peak windows within TSS ± 10 kb, from public HEK293 H3K27ac/H3K4me3/H3K4me1, HEK293T DNase-seq, PRO-cap and GEO GSE299613 H3K27ac peaks; 6.66 windows per gene) and ship prebuilt in `code/hek_universal_prediction/regions/` (`hek293_windows.bed`, `.fa`, 122,248 windows), with the 43-motif count matrix (`results/HEK293_count_matrix.npy`) and the Human Protein Atlas expression (`code/hek_hpa/`). Three models predict expression from the 43 motifs on the 18,221 genes with an HPA value, under the same 90/10 split:
* linear regression and random forest on the count matrix (`05_hek_linear_rf.py`, a minute);
* the 43-motif CNN-CR on the one-hot windows (`05a_build_cnn_inputs.py` then `05_hek_cnn.py`; GPU, about 8 min with the published configuration: dropout 0.3, doubled L2, learning rate 5·10⁻⁵, patience 15, up to 200 epochs, best-validation checkpoint).

Panel e places the three HEK293 values (stars) on the distributions of the same models over the 116 EpiMap biosamples.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
HK = f"{CODE}/hek_universal_prediction"; F4 = f"{REPO}/figures/fig4"
CNN_ARGS = "--motif_set universal --dropout 0.3 --reg_scale 2.0 --lr 5e-5 --patience 15 --epochs 200"     # the published HEK293 configuration
import numpy as np, pandas as pd
from IPython.display import Image, display
print("windows:", sum(1 for _ in open(f"{HK}/regions/hek293_windows.bed")), "| genes with windows:", sum(1 for _ in open(f"{HK}/regions/gene_window_counts.tsv")) - 1)
print("count matrix:", np.load(f"{HK}/results/HEK293_count_matrix.npy").shape, "| HPA expression rows:", sum(1 for _ in open(f"{CODE}/hek_hpa/HEK293_HPA_expression.tsv")) - 1)

## 9.1 Linear regression and random forest on the 43-motif counts

In [ ]:
run("python 05_hek_linear_rf.py", cwd=HK); print(open(f"{HK}/results/SUMMARY_linear_rf.txt").read())

## 9.2 The 43-motif CNN-CR (GPU)
`05a_build_cnn_inputs.py` one-hot-encodes the shipped windows (10 promoter slots per gene) and writes the log2(TPM+1) target; `05_hek_cnn.py` trains the motif-initialised CNN of notebook 05 with the published configuration and restores the best-validation epoch. GPU arithmetic is not bit-reproducible, so the held-out correlation varies by about ± 0.01 between runs.

In [ ]:
run("python 05a_build_cnn_inputs.py", cwd=HK)
if os.path.exists(f"{HK}/results/cnn/hek_universal_test_S.npy"): os.remove(f"{HK}/results/cnn/hek_universal_test_S.npy")   # the script skips existing results
run(f"python 05_hek_cnn.py {CNN_ARGS}", cwd=HK)

## 9.3 The three numbers and panel e
The published values are 0.659 (linear), 0.666 (random forest) and 0.728 (CNN-CR), at the 69th, 63rd and 65th percentile of the 116 biosamples.

In [ ]:
lr = np.load(f"{HK}/results/hek_linear_rf.npz"); cnn = float(np.load(f"{HK}/results/cnn/hek_universal_test_S.npy"))
ref = pd.read_csv(f"{CODE}/posthoc_analysis/results/linear_rf/correlations.csv"); cnn116 = np.array([float(np.load(f)) for f in sorted(glob.glob(f"{CODE}/posthoc_analysis/results/cnn/*_universal_test_S.npy"))])
for name, v, dist in (("linear", float(lr["linear_test"]), ref.linear_universal_test.values), ("random forest", float(lr["rf_test"]), ref.rf_universal_test.values), ("CNN-CR", cnn, cnn116)):
    print(f"  HEK293 {name:14s} held-out Spearman {v:.3f}   ({(dist < v).mean()*100:.0f}th percentile of {len(dist)} EpiMap biosamples, median {np.median(dist):.3f})")
run("python -c \"import make_fig4_panels as P; P.panel_4f()\"", cwd=F4); display(Image(f"{F4}/fig4f_hek_heldout.png", width=300))